# 03 · Deep Researcher Agent

**Goal:** Take the ranked `GapReport` from the Gap Analysis agent and produce **personalized learning pathways** — courses, projects, checklists — grounded in current web knowledge via Tavily search.

This notebook implements the architecture from `Documentation/Deep Researcher Agentic System.pdf`:

```
        ┌────────────────── Deep Researcher LangGraph ──────────────────┐
Gap ───▶│  Researcher (plans + critiques) ⇄  Tavily Search (online)     │
Report  │                       │                                       │
        │                       ▼                                       │
        │            Structuring Agent (recursive Pydantic)             │
        └────────────────────────┬──────────────────────────────────────┘
                                 ▼
                        Learning Pathways
```

### Key design decisions
- **Researcher ↔ Tavily loop** with a `continue | stop` router. Budget is measured in search iterations (default 3) AND in "gathered notes" minimum.
- **Critic step inside the researcher.** After each Tavily call the researcher decides: (a) do I have enough, or (b) what's missing — and drafts the next query. This is what separates a "search once and summarize" pipeline from a deep researcher.
- **Structuring agent** uses a **recursive Pydantic model** (`Pathway` contains `Milestone` which contains `Resource`) so Gemini emits a tree in one call with automatic validation.
- Entire thing lives in a LangGraph subgraph with shared state — drops straight into the supervisor.


## 0. Setup

In [ ]:
# !pip install -q langchain langchain-google-genai langchain-community tavily-python langgraph pydantic python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path("../backend/.env").resolve())
assert os.getenv("GOOGLE_API_KEY")
assert os.getenv("TAVILY_API_KEY")
print("OK")

## 1. Tools — Tavily search

`TavilySearchResults` returns `[{url, content, ...}]`. We'll keep `search_depth="advanced"` for research-grade results and `max_results=5` to stay inside free-tier budgets.


In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)

# Smoke test
sample = tavily.invoke({"query": "best course to learn PyTorch in 2025"})
print(f"{len(sample)} results")
print(sample[0]["url"], "—", sample[0]["content"][:120])

## 2. Output schema — recursive Pydantic model

A `Pathway` is a tree: `Pathway → Milestone[] → Resource[]`. Gemini emits this in one structured call at the end of the research loop.


In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

class Resource(BaseModel):
    title: str
    kind: Literal["course", "doc", "video", "article", "project"]
    provider: str = Field(description="e.g. Coursera, YouTube, official docs, fast.ai")
    url: str
    why: str = Field(description="1 sentence — why this resource for this milestone")

class Milestone(BaseModel):
    phase: Literal["Foundations", "Intermediate", "Advanced"]
    skill: str = Field(description="Gap skill being closed")
    estimated_weeks: int
    objective: str = Field(description="What the learner will be able to do after this milestone")
    resources: List[Resource] = Field(default_factory=list)
    checklist: List[str] = Field(description="3-5 concrete, checkable items")
    mini_project: Optional[str] = Field(None, description="Hands-on project suggestion")

class Pathway(BaseModel):
    target_role: str
    milestones: List[Milestone]
    rationale: str = Field(description="2-3 sentence reasoning on why the ordering works")

## 3. Researcher state + nodes

State fields:
- `gaps`: the incoming `GapReport.gaps` list
- `target_role`
- `notes`: list[str] — accumulated search findings, one entry per round
- `last_query`: the query just executed (for logging)
- `iteration`: round counter
- `max_iter`: hard stop
- `pathway`: final output

Nodes:
- `planner` — given gaps + notes so far, emit the next search query. On iteration 0 the query is broad; later iterations drill into whatever's still missing.
- `search` — call Tavily with `last_query`, append trimmed results to `notes`.
- `critic` — decide "continue" (need more notes) or "structure" (enough).
- `structurer` — one Gemini call that emits the full `Pathway`.


In [ ]:
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel as PydBase

gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

# Minimal shape of an incoming gap (mirrors notebook 02's Gap).
class GapIn(PydBase):
    skill: str
    category: str
    relevance: int
    difficulty: str
    prerequisites: List[str]
    why: str

class ResearcherState(TypedDict, total=False):
    gaps: List[GapIn]
    target_role: str
    notes: List[str]
    last_query: str
    iteration: int
    max_iter: int
    pathway: Pathway

### Planner

The planner produces one query per iteration. Prompt it to (a) cover the highest-relevance gap first, (b) drill into gaps the previous searches didn't cover.


In [ ]:
class NextQuery(PydBase):
    query: str = Field(description="The single best next search query — concise, 6-15 words")
    rationale: str = Field(description="Why this query, 1 sentence")

plan_prompt = ChatPromptTemplate.from_template('''You are a research planner for career-skill learning pathways.

TARGET ROLE: {target_role}

OUTSTANDING GAPS (skill, relevance, difficulty):
{gaps}

NOTES GATHERED SO FAR (from previous searches, newest last):
{notes}

Propose ONE next web search query that will most improve the pathway.
Rules:
- Iteration 0: start with the highest-relevance gap and look for current (2025+) top courses/docs.
- Later iterations: cover gaps not yet represented in the notes, or drill deeper on a specific resource.
- Prefer concrete queries ("best PyTorch course 2025 for production ML") over vague ones.''')

planner = plan_prompt | gemini.with_structured_output(NextQuery)

def node_plan(state: ResearcherState) -> ResearcherState:
    gaps_str = "\n".join(f"- {g.skill} (rel={g.relevance}, {g.difficulty})" for g in state["gaps"])
    notes_str = "\n".join(f"[{i}] {n[:300]}" for i, n in enumerate(state.get("notes", []))) or "(none yet)"
    nxt: NextQuery = planner.invoke({
        "target_role": state["target_role"],
        "gaps": gaps_str,
        "notes": notes_str,
    })
    print(f"iter {state.get('iteration', 0)} → plan: {nxt.query}  ({nxt.rationale})")
    return {"last_query": nxt.query, "iteration": state.get("iteration", 0) + 1}

### Search node

In [ ]:
def node_search(state: ResearcherState) -> ResearcherState:
    results = tavily.invoke({"query": state["last_query"]})
    # Compress each result into a single line: url + first 500 chars.
    summary = "\n".join(f"{r['url']} :: {r.get('content','')[:500]}" for r in results)
    notes = state.get("notes", []) + [f"Q={state['last_query']}\n{summary}"]
    return {"notes": notes}

### Critic / router

Given the notes gathered, do we have enough? The router is a simple LLM classifier. We also hard-cap at `max_iter` so Tavily doesn't burn quota on a degenerate loop.


In [ ]:
class CriticOut(PydBase):
    decision: Literal["continue", "structure"]
    rationale: str

critic_prompt = ChatPromptTemplate.from_template('''You are a research critic.

TARGET ROLE: {target_role}
GAPS TO COVER:
{gaps}

NOTES GATHERED (newest last):
{notes}

Decide:
- "continue" if the notes don't yet give us enough to recommend concrete resources for EVERY gap.
- "structure" if we can now write a high-quality learning pathway with real links for every gap.''')

critic = critic_prompt | gemini.with_structured_output(CriticOut)

def node_critic_route(state: ResearcherState) -> str:
    # Hard stop on max iterations.
    if state.get("iteration", 0) >= state.get("max_iter", 3):
        print("→ max_iter reached, forcing structure")
        return "structure"
    gaps_str = "\n".join(f"- {g.skill}" for g in state["gaps"])
    notes_str = "\n".join(f"[{i}] {n[:500]}" for i, n in enumerate(state.get("notes", [])))
    out: CriticOut = critic.invoke({
        "target_role": state["target_role"],
        "gaps": gaps_str,
        "notes": notes_str,
    })
    print(f"→ critic: {out.decision} — {out.rationale}")
    return out.decision

### Structurer

One call. Takes gaps + all notes and emits the full `Pathway` tree.


In [ ]:
struct_prompt = ChatPromptTemplate.from_template('''You are a senior learning-path designer.

TARGET ROLE: {target_role}

USER'S SKILL GAPS (ordered by relevance, prerequisites first):
{gaps}

RESEARCH NOTES (URLs + extracts from authoritative sources):
{notes}

Produce a Pathway with:
- ONE milestone per gap, in prerequisite-respecting order.
- Each milestone in an appropriate phase (Foundations / Intermediate / Advanced).
- 2-4 real resources per milestone, PREFERRING URLs that appear in the research notes.
- A 3-5 item `checklist` of concrete actions.
- A `mini_project` that exercises the skill in context of the target role.
- `rationale`: why this ordering works for this user.''')

structurer = struct_prompt | gemini.with_structured_output(Pathway)

def node_structure(state: ResearcherState) -> ResearcherState:
    gaps_str = "\n".join(
        f"- {g.skill} (rel={g.relevance}, {g.difficulty}, prereqs={g.prerequisites})" for g in state["gaps"]
    )
    notes_str = "\n".join(state.get("notes", []))
    pathway: Pathway = structurer.invoke({
        "target_role": state["target_role"],
        "gaps": gaps_str,
        "notes": notes_str,
    })
    return {"pathway": pathway}

## 4. Wire up the LangGraph

In [ ]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(ResearcherState)
g.add_node("plan",      node_plan)
g.add_node("search",    node_search)
g.add_node("structure", node_structure)

g.add_edge(START, "plan")
g.add_edge("plan", "search")
g.add_conditional_edges(
    "search",
    node_critic_route,            # returns "continue" or "structure"
    {"continue": "plan", "structure": "structure"},
)
g.add_edge("structure", END)

researcher = g.compile()

try:
    from IPython.display import Image, display
    display(Image(researcher.get_graph().draw_mermaid_png()))
except Exception:
    print(researcher.get_graph().draw_mermaid())

## 5. Run it end-to-end

Feed in sample gaps (would come from notebook 02) and see the loop do plan → search → [critic decides] → plan → search → ... → structure.


In [ ]:
# Sample gaps (normally produced by 02_gap_analysis_rag.ipynb)
sample_gaps = [
    GapIn(skill="Transformers",      category="ML",  relevance=95, difficulty="Hard",
          prerequisites=["PyTorch"],
          why="Core architecture for modern NLP/LLM work; expected of every ML engineer."),
    GapIn(skill="PyTorch",           category="ML",  relevance=92, difficulty="Medium",
          prerequisites=["Python", "Linear Algebra"],
          why="Primary deep learning framework in production ML roles."),
    GapIn(skill="MLflow",            category="MLOps", relevance=75, difficulty="Easy",
          prerequisites=["Python"],
          why="Standard for experiment tracking + model registry."),
]

state_in = {
    "gaps": sample_gaps,
    "target_role": "ml-engineer",
    "notes": [],
    "iteration": 0,
    "max_iter": 3,  # keep this small for Tavily quota
}

final = researcher.invoke(state_in, {"recursion_limit": 20})
path: Pathway = final["pathway"]

In [ ]:
print(f"Target role: {path.target_role}")
print(f"Rationale:   {path.rationale}\n")
for i, m in enumerate(path.milestones, 1):
    print(f"{i}. [{m.phase:13s}] {m.skill}  ({m.estimated_weeks}w)")
    print(f"   objective: {m.objective}")
    for r in m.resources:
        print(f"     · [{r.kind}] {r.title} — {r.provider}")
        print(f"       {r.url}")
    print(f"   checklist: {m.checklist}")
    print(f"   project:   {m.mini_project}\n")

## 6. Notes for the next engineer

- **Iteration budget.** `max_iter=3` for demos. Bump to 5-6 in production; keep the critic honest so most queries terminate early on simple profiles.
- **Tavily cost control.** Each loop spends 1 Tavily call. With 5 results × 3 iterations you typically hit ~15 URLs in the notes. Cache per (target_role, gaps_hash) in Redis.
- **Note compression.** Right now we truncate each result at 500 chars. If you swap in larger context, consider an additional summarizer node between `search` and `critic` that distills notes into bullet points — keeps the structurer prompt short.
- **Grounding.** Tell the structurer to prefer URLs that appear in notes (already in the prompt). In production add a post-hoc validator: for every emitted URL, assert it was in the notes. If not, either drop the resource or flag it.
- **Critic drift.** In long runs the critic can ping-pong on "continue". Safeguard: if the same query is proposed twice in a row, force "structure".
- **Downstream.** The `Pathway` is either shipped directly to the frontend or persisted to Supabase as `milestones` rows (see `sql/001_schema.sql`).
